In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np

# ==============================================================================
# CELL 1: LOAD GZT DATASET & FILTER FOR CARLOW (CWLA)
# ==============================================================================
print("[1/4] Loading GZT dataset...")
# Load GZT file (Replace with your actual local file path)
gzt_path = "Datasets/GZT_Current_Plan_7352405634776391600.geojson"  
gzt_full = gpd.read_file(gzt_path)

# Ensure Irish Transverse Mercator Projection (EPSG:2157) for meter/hectare calculations
if gzt_full.crs != "EPSG:2157":
    gzt_full = gzt_full.to_crs(epsg=2157)

# Flexible filtering for Carlow County Council across code and name variations
cwla_mask = (
    (gzt_full['LA_CODE'].astype(str).str.strip().str.upper() == 'CWLA') |
    (gzt_full['LA_NAME'].astype(str).str.contains('Carlow', case=False, na=False))
)
gzt_cwla = gzt_full[cwla_mask].copy()

print(f"Total GZT features loaded: {len(gzt_full)}")
print(f"Filtered GZT features for CWLA: {len(gzt_cwla)}")

[1/4] Loading GZT dataset...
Total GZT features loaded: 86686
Filtered GZT features for CWLA: 3402


In [ ]:
# ==============================================================================
# CELL 2: FEATURE CLEANING & GEOMETRY PROCESSING
# ==============================================================================
print("[2/4] Processing geometry and engineering features...")

# 1. Standardize Local Authority Code
gzt_cwla['LOCAL_AUTHORITY'] = 'CWLA'

# 2. Compute Site Area in Hectares from EPSG:2157 geometry area (m² / 10,000)
gzt_cwla['SITE_AREA'] = gzt_cwla.geometry.area / 10000.0

# 3. Clean and standardize GZT codes (handling N/A or missing values)
gzt_cwla['ZONE_GZT'] = gzt_cwla['ZONE_GZT'].fillna('UNZ').astype(str).str.strip()

# 4. Extract Polygon Centroids (ITM coordinates in meters)
gzt_cwla['centroid_x'] = gzt_cwla.geometry.centroid.x
gzt_cwla['centroid_y'] = gzt_cwla.geometry.centroid.y

# 5. Remove tiny geometry artifacts / slivers (< 0.01 Hectares)
gzt_cwla_clean = gzt_cwla[gzt_cwla['SITE_AREA'] >= 0.01].copy()

print(f"Cleaned CWLA dataset records: {len(gzt_cwla_clean)}")

[2/4] Processing geometry and engineering features...
Cleaned CWLA dataset records: 3310


In [ ]:
# ==============================================================================
# CELL 3: EXTRACT & STRUCTURE A SAMPLE DATASET
# ==============================================================================
print("[3/4] Creating structured sample...")

# Select key modeling and metadata columns
columns_to_keep = [
    'OBJECTID', 
    'LOCAL_AUTHORITY', 
    'LA_NAME', 
    'ZONE_GZT', 
    'GZT_DESC', 
    'ZONE_ORIG', 
    'SITE_AREA', 
    'centroid_x', 
    'centroid_y', 
    'geometry'
]

# Keep available columns
selected_cols = [c for c in columns_to_keep if c in gzt_cwla_clean.columns]
gzt_cwla_sample = gzt_cwla_clean[selected_cols].copy().reset_index(drop=True)

print(f"Sample dataset shape: {gzt_cwla_sample.shape}")

[3/4] Creating structured sample...
Sample dataset shape: (200, 10)


In [ ]:
# ==============================================================================
# CELL 4: SUMMARY BREAKDOWN & EXPORT
# ==============================================================================
print("[4/4] GZT CWLA Sample Summary:")

print("\n--- Top Zoning Breakdown (ZONE_GZT) ---")
print(gzt_cwla_sample['ZONE_GZT'].value_counts())

print("\n--- Site Area Statistics (Hectares) ---")
print(gzt_cwla_sample['SITE_AREA'].describe())

# Save sample locally as a GeoJSON file
output_filename = "Datasets/GZT_Samples/CWLA_GZT_Sample.geojson"
gzt_cwla_sample.to_file(output_filename, driver="GeoJSON")
print(f"\nSample saved successfully to '{output_filename}'!")

[4/4] GZT CWLA Sample Summary:

--- Top Zoning Breakdown (ZONE_GZT) ---
ZONE_GZT
R2      91
G1      62
M2      12
S5      10
R1       6
P1       5
C2.1     4
M3       3
C6       3
O1       2
C5       1
N2.2     1
Name: count, dtype: int64

--- Site Area Statistics (Hectares) ---
count    200.000000
mean       0.829031
std        1.373619
min        0.012316
25%        0.099496
50%        0.275793
75%        0.824353
max        6.736688
Name: SITE_AREA, dtype: float64

Sample saved successfully to 'Datasets/GZT_Samples/CWLA_GZT_Sample.geojson'!
